In [1]:
import pandas as pd
import spacy
from collections import Counter
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import numpy as np
from tqdm import tqdm



In [2]:
spacy.prefer_gpu()
# Load the dataset
dataset_path = "Combined.csv"
df_test = pd.read_csv(dataset_path)

# Load spaCy model
nlp = spacy.load("en_core_web_trf")

/home/shaisu/.conda/envs/ds_env/lib/python3.11/site-packages/thinc/shims/pytorch.py:253: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filel

In [3]:

# Function to extract the most common location from text
def extract_most_common_location(text, pbar):

    doc = nlp(text)
    locations = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
    locations = ["United States" if loc in United_states_synonyms else loc for loc in locations]

    if locations:
        location_counts = Counter(locations)
        most_common_location = location_counts.most_common(1)[0][0]
        if "@" in most_common_location:
            return "No location"
        # if most common location is "United States" - take the second most common if there is one
        if most_common_location == "United States":
            if len(location_counts) > 1:
                most_common_location = location_counts.most_common(2)[1][0]
            else:
                # If no other locations are available, keep "United States"
                return most_common_location

        return  most_common_location
    pbar.update()
    return "No location"


# Function to find geocode with caching
def find_Geocode(location, geolocator, cache):
    if location == "No location":
        return 0, 0

    if location in cache:
        return cache[location]
    try:
        loc = geolocator.geocode(location)
        if loc:
            cache[location] = (loc.latitude, loc.longitude)
            return loc.latitude, loc.longitude
    except GeocoderTimedOut:
        return None
    cache[location] = (np.nan, np.nan)
    return 0, 0

In [4]:
df_test.shape

(44898, 5)

In [5]:
df_test.head()

,title,text,subject,date,label
0,That Time An Ohio Ammosexual 2nd Amendmented ...,"Ammosexuals also tend to be anti-gay bigots, s...",News,"January 23, 2016",0
1,"Turkey to allow muftis to conduct weddings, sp...",ANKARA (Reuters) - Turkey s parliament approve...,worldnews,"October 20, 2017",1
2,Israel's right wing has grand plans for Trump era,JERUSALEM (Reuters) - Israel’s right wing has ...,politicsNews,"January 19, 2017",1
3,EPIC! RAND PAUL Laughs at CNN’s Climate Hyster...,This is so good! Rand Paul knows his climate t...,politics,"Jun 2, 2017",0
4,Episode #154 – SUNDAY WIRE: ‘The Pro-War Left?...,Episode #154 of SUNDAY WIRE SHOW resumes this...,US_News,"September 25, 2016",0


In [6]:
United_states_synonyms = ["U.S","America","States","US"]

In [7]:
df_test_500 = df_test.head(500).copy()

In [8]:
df_test_500

,title,text,subject,date,label
0,That Time An Ohio Ammosexual 2nd Amendmented ...,"Ammosexuals also tend to be anti-gay bigots, s...",News,"January 23, 2016",0
1,"Turkey to allow muftis to conduct weddings, sp...",ANKARA (Reuters) - Turkey s parliament approve...,worldnews,"October 20, 2017",1
2,Israel's right wing has grand plans for Trump era,JERUSALEM (Reuters) - Israel’s right wing has ...,politicsNews,"January 19, 2017",1
3,EPIC! RAND PAUL Laughs at CNN’s Climate Hyster...,This is so good! Rand Paul knows his climate t...,politics,"Jun 2, 2017",0
4,Episode #154 – SUNDAY WIRE: ‘The Pro-War Left?...,Episode #154 of SUNDAY WIRE SHOW resumes this...,US_News,"September 25, 2016",0
...,...,...,...,...,...
495,LOL! #VeryFakeNewsCNN Claims Anderson Cooper’s...,After Roy Moore s ugly loss in the Alabama Sen...,politics,"Dec 13, 2017",0
496,Soccer star Weah to face vice president in Lib...,MONROVIA (Reuters) - Former soccer star George...,worldnews,"October 19, 2017",1
497,U.S. House Judiciary Democrats ask FBI to inve...,WASHINGTON (Reuters) - Democratic members of t...,politicsNews,"March 2, 2017",1
498,"Charity shop customers leave empty-handed, ref...",LONDON (Reuters) - Customers at a new charity ...,worldnews,"December 1, 2017",1


In [ ]:
with tqdm(total=len(df_test_500), desc="extract_most_common_location") as pbar:
    # Apply function to text column of the first 10 rows
    df_test['location'] = df_test['text'].apply(lambda p: extract_most_common_location(p, pbar))


extract_most_common_location:   0%|                                                                                                                                                                                                                     | 0/500 [00:00<?, ?it/s]/home/shaisu/.conda/envs/ds_env/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
extract_most_common_location:   4%|████████▏                                                                                                                                                                                                   | 20/500 [01:54<41:57,  5.25s/it]

In [ ]:




# Initialize geocoder and cache
geolocator = Nominatim(user_agent="Yuval", timeout=10)
cache = {}

# Geocode locations
latitude = []
longitude = []

with tqdm(total=len(df_test), desc="find_Geocode") as pbar:
    
    for i, location in enumerate(df_test["location"]):
        lat, lon = find_Geocode(location, geolocator, cache)
        if lat == 0 or lon == 0:
            # Update the location in the DataFrame if geocoding fails
            df_test.at[i, "location"] = "No location"
        latitude.append(lat)
        longitude.append(lon)
        pbar.update()

# Add lat, long to dataframe
df_test["Latitude"] = latitude
df_test["Longitude"] = longitude

# Save the results to an Excel file
output_path = "Combined_with_lat_long.csv"
df_test.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")